# Fundamental Stock Data
In this homework we will guide you through how to download free historical fundamentals for stocks. This data can be used to construct value and other fundamental strategies. There will be no "solution" for this homework as it is more of a guide.

### Packages
We will be using the <a href='https://pypi.org/project/simfin/' target="_blank" >simfin</a> and  <a href='https://pypi.org/project/yfinance/' target="_blank" >yfinance</a> packages. Install them using "pip install simfin" and "pip install yfinance" if you haven't already and import them as below.

In [2]:
import yfinance as yf 
import simfin as sf

### Get Your SimFin API Key

First you must obtain an API key. Make an account at https://simfin.com. Confirm your account via email and then head to https://simfin.com/data/api to obtain your api key.

In [6]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv('SIMFIN_API_KEY')

In [9]:
# set your api key here

sf.config.set_api_key(api_key=api_key)

### Set Your SimFin Data Directory

Simfin requires you to set a data directory where simfin data will be downloaded. Downloaded data is used for faster retrieval in the future. The default location is a folder named simfin_data in the home directory.

In [10]:
# set simfin_data
sf.set_data_dir('~/simfin_data/')

### Download historical financial data from simfin.
The three financial statements containing fundamental data are the quarterly income, balance sheet and cash flows statements. Data from these statements can be loaded for all us tickers from simfin as below.

In [11]:
income = sf.load_income(variant='quarterly', market='us')

balance_sheet = sf.load_balance(variant='quarterly', market='us')

cash_flow = sf.load_cashflow(variant='quarterly', market='us')

Dataset "us-income-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-balance-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!


Below we observe the shape of the data.

All of the results are multi-index dataframes where the row is (ticker,date) and the column is a financial statement item. 

One of the important columns for backtesting is the Publish Date. This is the date the information was available to the public. 

For backtesting, we can assume the simfin data is available 1 business day after the Publish Date, since companies sometimes publish after market close or because we may not always be able to get the data immediately on publication for trading, depending on which data vendor you use. 

Because there are restatements to financial statements, even this will not be a fully "point-in-time" backtest where we only use information available at the time. It should be a reasonable approximation in many cases, however.

In [12]:
cash_flow.head()

SimFinId Currency  Fiscal Year Fiscal Period Publish Date  \
Ticker Report Date                                                              
A      2020-10-31      45846      USD         2020            Q4   2020-12-18   
       2021-01-31      45846      USD         2021            Q1   2021-03-02   
       2021-04-30      45846      USD         2021            Q2   2021-06-01   
       2021-07-31      45846      USD         2021            Q3   2021-09-01   
       2021-10-31      45846      USD         2021            Q4   2021-12-17   

                   Restated Date  Shares (Basic)  Shares (Diluted)  \
Ticker Report Date                                                   
A      2020-10-31     2021-09-01     308000000.0       311000000.0   
       2021-01-31     2022-03-03     306000000.0       309000000.0   
       2021-04-30     2022-03-03     306000000.0       306000000.0   
       2021-07-31     2022-05-31     303000000.0       306000000.0   
       2021-10-31     2022-09-01     305000000.0       307000000.0   

                    Net Income/Starting Line  Depreciation & Amortization  \
Ticker Report Date                                                          
A      2020-10-31                222000000.0                   76000000.0   
       2021-01-31                288000000.0                   76000000.0   
       2021-04-30                216000000.0                   77000000.0   
       2021-07-31                264000000.0                   84000000.0   
       2021-10-31                442000000.0                   84000000.0   

                    ...  Net Cash from Operating Activities  \
Ticker Report Date  ...                                       
A      2020-10-31   ...                         377000000.0   
       2021-01-31   ...                         238000000.0   
       2021-04-30   ...                         472000000.0   
       2021-07-31   ...                         334000000.0   
       2021-10-31   ...                         441000000.0   

                    Change in Fixed Assets & Intangibles  \
Ticker Report Date                                         
A      2020-10-31                            -27000000.0   
       2021-01-31                            -41000000.0   
       2021-04-30                            -31000000.0   
       2021-07-31                            -54000000.0   
       2021-10-31                            -61000000.0   

                    Net Change in Long Term Investment  \
Ticker Report Date                                       
A      2020-10-31                                  NaN   
       2021-01-31                                  NaN   
       2021-04-30                                  NaN   
       2021-07-31                                  NaN   
       2021-10-31                           12000000.0   

                    Net Cash from Acquisitions & Divestitures  \
Ticker Report Date                                              
A      2020-10-31                                         NaN   
       2021-01-31                                         NaN   
       2021-04-30                                -547000000.0   
       2021-07-31                                         0.0   
       2021-10-31                                         0.0   

                    Net Cash from Investing Activities  Dividends Paid  \
Ticker Report Date                                                       
A      2020-10-31                          -27000000.0     -55000000.0   
       2021-01-31                          -42000000.0     -59000000.0   
       2021-04-30                         -587000000.0     -59000000.0   
       2021-07-31                          -61000000.0     -59000000.0   
       2021-10-31                          -59000000.0     -59000000.0   

                    Cash from (Repayment of) Debt  \
Ticker Report Date                                  
A      2020-10-31                      35000000.0   
       2021-01-31

Let's observe which columns are available to us below from each financial statement.

In [13]:
# observe available income columns
income.columns

Index(['SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date',
       'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Revenue',
       'Cost of Revenue', 'Gross Profit', 'Operating Expenses',
       'Selling, General & Administrative', 'Research & Development',
       'Depreciation & Amortization', 'Operating Income (Loss)',
       'Non-Operating Income (Loss)', 'Interest Expense, Net',
       'Pretax Income (Loss), Adj.', 'Abnormal Gains (Losses)',
       'Pretax Income (Loss)', 'Income Tax (Expense) Benefit, Net',
       'Income (Loss) from Continuing Operations',
       'Net Extraordinary Gains (Losses)', 'Net Income',
       'Net Income (Common)'],
      dtype='object')

In [14]:
# observe available balance_sheet columns
balance_sheet.columns

Index(['SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date',
       'Restated Date', 'Shares (Basic)', 'Shares (Diluted)',
       'Cash, Cash Equivalents & Short Term Investments',
       'Accounts & Notes Receivable', 'Inventories', 'Total Current Assets',
       'Property, Plant & Equipment, Net',
       'Long Term Investments & Receivables', 'Other Long Term Assets',
       'Total Noncurrent Assets', 'Total Assets', 'Payables & Accruals',
       'Short Term Debt', 'Total Current Liabilities', 'Long Term Debt',
       'Total Noncurrent Liabilities', 'Total Liabilities',
       'Share Capital & Additional Paid-In Capital', 'Treasury Stock',
       'Retained Earnings', 'Total Equity', 'Total Liabilities & Equity'],
      dtype='object')

In [15]:
# observe available cash_flow columns
cash_flow.columns

Index(['SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date',
       'Restated Date', 'Shares (Basic)', 'Shares (Diluted)',
       'Net Income/Starting Line', 'Depreciation & Amortization',
       'Non-Cash Items', 'Change in Working Capital',
       'Change in Accounts Receivable', 'Change in Inventories',
       'Change in Accounts Payable', 'Change in Other',
       'Net Cash from Operating Activities',
       'Change in Fixed Assets & Intangibles',
       'Net Change in Long Term Investment',
       'Net Cash from Acquisitions & Divestitures',
       'Net Cash from Investing Activities', 'Dividends Paid',
       'Cash from (Repayment of) Debt', 'Cash from (Repurchase of) Equity',
       'Net Cash from Financing Activities', 'Net Change in Cash'],
      dtype='object')

Below we observe the number of tickers available for each statement and the start and end dates.

In [16]:
def describe_data(data, data_name):
    size = len(set(data.index.get_level_values(0)))
    start_dt = data.index.get_level_values(1).min().strftime('%Y%m%d')
    end_dt = data.index.get_level_values(1).max().strftime('%Y%m%d')
    print (f'{data_name}: {size} tickers. Date range: {start_dt} to {end_dt}')

describe_data(income ,'Income Data')
describe_data(balance_sheet ,'Balance Sheet Data')
describe_data(cash_flow ,'Cash Flow Data')

Income Data: 3781 tickers. Date range: 20200831 to 20250630
Balance Sheet Data: 3782 tickers. Date range: 20200830 to 20250630
Cash Flow Data: 3781 tickers. Date range: 20200831 to 20250630


### Yahoo Finance Data
If you notice above, the simfin data ends a year ago. They only have 1 year lagged data. To get the most recent financial statement information, you can use yahoo finance.

First you must obtain a yfinance.Ticker object for your desired ticker.

In [17]:
yf_ticker = yf.Ticker('AAPL')

Download information from the 3 financial statements as below.

In [18]:
# Get the quarterly cash flow statements from yfinance

income_yf = yf_ticker.quarterly_financials

cash_flow_yf = yf_ticker.quarterly_cashflow

balance_sheet = yf_ticker.quarterly_balance_sheet

In [19]:
income_yf

,2026-03-31,2025-12-31,2025-09-30,2025-06-30,2025-03-31
Tax Effect Of Unusual Items,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Tax Rate For Calcs,1.750000e-01,1.750000e-01,1.627240e-01,1.640000e-01,1.545550e-01
Normalized EBITDA,3.932400e+10,5.406600e+10,3.555400e+10,3.103200e+10,3.225000e+10
Net Income From Continuing Operation Net Minority Interest,2.957800e+10,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10
Reconciled Depreciation,3.439000e+09,3.214000e+09,3.127000e+09,2.830000e+09,2.661000e+09
Reconciled Cost Of Revenue,5.640300e+10,7.452500e+10,5.412500e+10,5.031800e+10,5.049200e+10
EBITDA,3.932400e+10,5.406600e+10,3.555400e+10,3.103200e+10,3.225000e+10
EBIT,3.588500e+10,5.085200e+10,3.242700e+10,2.820200e+10,2.958900e+10
Normalized Income,2.957800e+10,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10
Net Income From Continuing And Discontinued Operation,2.957800e+10,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10
